# Docking pose 2

This copy writes to `MyDrive/PABPC1_MMGBSA_pose_02`. Use the three notebooks in this same pose folder together. The original notebooks use `MyDrive/PABPC1_MMGBSA`. This copy can reuse the original system’s workbooks; select this pose’s own PDB when prompted.

The input checker requires RNA chain E containing 30 adenines and protein chain A with 400 residues whose first residue IDs are 8, 9, and 11; it rebuilds the missing Met10. A different chain layout, sequence, or protein model needs adaptation before use. The repaired pose and calculated scores remain exploratory.

# PABPC1–RNA minimization and GBn1 screening

**Research prototype.** This workflow creates modeled complexes, not experimentally verified structures. It uses 15 nt of 3′UTR followed by A15 and the supplied human PABPC1 model. A preliminary geometric repair is performed before the production minimization. Do not interpret the resulting single-structure scores as measured affinities or deadenylation rates.

The force field is Amber ff14SB for protein and OL3 for RNA, with bundled modXNA libraries and frcmod parameters. Both RNA ends use a consistent modXNA terminal charge convention. His residues use Amber's default HIE assignment and require scientific review. The truncated 30-nt RNA has an assumed 5′-OH at position −15, creating an artificial terminal environment. A TIP3P box extends 8 Å beyond the solute, is neutralized, and contains approximately 0.09 M added KCl. This approximates Müller’s potassium-acetate buffer and does not reproduce Mg²⁺, glycerol, protein additives, or full-length PABPC1. Minimization alone has no thermodynamic temperature and does not reproduce the 37 °C assay.

OpenMM on CUDA uses L-BFGS, not the steepest-descent/conjugate-gradient algorithm of Chen et al. It runs three stages with protein backbone restraints of 50, 10 and 0 kcal mol⁻¹ Å⁻² and at most 5,000 iterations each. The related score notebook uses GBn1, Bondi radii, internal dielectric 2, LCPO and a single minimized structure; no conformational entropy is included. Bondi radii for sulfur and fluorine are generic defaults; GBn1 has not been specifically validated here for these modified protein–RNA complexes.

Your L4 must be selected in **Runtime → Change runtime type** for minimization. Run cells in order. The case-by-case logs show elapsed time and a rough remaining-time estimate; availability, setup, input preparation, GPU speed and scoring overhead can change that estimate. Pro+ does not guarantee a particular deadline. Compressed full inputs and final coordinates are stored on Drive, with failure and convergence checks. Score only cases that pass all checks.

Sources: [Chen et al. 2018](https://rnajournal.cshlp.org/content/24/9/1183), [Müller et al. 2026 preprint](https://doi.org/10.64898/2026.06.05.728831), [modXNA](https://doi.org/10.1021/acs.jctc.4c01164), [OpenMM minimizer](https://docs.openmm.org/development/api-python/generated/openmm.openmm.LocalEnergyMinimizer.html).


This notebook can create the shared template if needed, then runs the 27 controls and 760 variant design rows, including eight duplicates. Use the same Drive folder in all three notebooks.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/PABPC1_MMGBSA_pose_02')  # Change consistently in all three notebooks if desired.
INPUTS = ROOT / 'inputs'
INPUTS.mkdir(parents=True, exist_ok=True)
print('Shared project folder:', ROOT)


In [ ]:
from google.colab import files
import hashlib
import shutil

# Use the same workbooks as the original system when they are available.
workbooks = [
    'Muller_benchmark.xlsx',
    'Muller_Position_Top_Two_Chemical_Modification_Library.xlsx',
]
baseline_inputs = Path('/content/drive/MyDrive/PABPC1_MMGBSA/inputs')
for name in workbooks:
    destination = INPUTS / name
    source = baseline_inputs / name
    if not destination.exists() and source.is_file():
        shutil.copy2(source, destination)
        print('Copied workbook from the original system:', name)

missing_workbooks = [name for name in workbooks if not (INPUTS / name).is_file()]
pdb_destination = INPUTS / 'temp_complex.pdb'
need_pdb = not pdb_destination.is_file()
if missing_workbooks or need_pdb:
    if missing_workbooks:
        print('Select these workbooks:', ', '.join(missing_workbooks))
    if need_pdb:
        print('Select exactly one PDB file for THIS pose. Its filename can be different.')
    uploaded = files.upload()
    for name in missing_workbooks:
        if name in uploaded:
            (INPUTS / name).write_bytes(uploaded[name])
    if need_pdb:
        candidates = [name for name in uploaded if name.lower().endswith('.pdb')]
        if len(candidates) != 1:
            raise ValueError('Select exactly one .pdb file for this pose, then rerun the cell.')
        pdb_destination.write_bytes(uploaded[candidates[0]])
        print('Using PDB:', candidates[0], 'as', pdb_destination)

missing = [name for name in workbooks + ['temp_complex.pdb'] if not (INPUTS / name).is_file()]
if missing:
    raise FileNotFoundError('Still missing: ' + ', '.join(missing))
print('Input folder:', INPUTS)
print('PDB SHA-256:', hashlib.sha256(pdb_destination.read_bytes()).hexdigest())
print('All three inputs are available.')


In [ ]:
import os, io, tarfile, urllib.request, subprocess, zipfile, base64
from pathlib import Path

RUNTIME_DIR = Path('/content/pabpc1_pose02_runtime')  # Scratch space; Drive contains persistent results.
ENV_DIR = RUNTIME_DIR / 'environment'
ASSETS = RUNTIME_DIR / 'assets'
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
# The archive contains code, 23 modXNA libraries, the modXNA frcmod file,
# and the ModeRNA source and licenses. Amber ff14SB/OL3 come from AmberTools.
assets_b64 = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(assets_b64))) as z:
    z.extractall(ASSETS)

mamba = RUNTIME_DIR / 'bin' / 'micromamba'
if not mamba.exists():
    mamba.parent.mkdir(parents=True, exist_ok=True)
    archive = RUNTIME_DIR / 'micromamba.tar.bz2'
    urllib.request.urlretrieve('https://micro.mamba.pm/api/micromamba/linux-64/latest', archive)
    with tarfile.open(archive, 'r:bz2') as t:
        member = t.extractfile('bin/micromamba')
        if member is None: raise RuntimeError('micromamba archive is missing its executable')
        mamba.write_bytes(member.read())
    mamba.chmod(0o755)
if not (ENV_DIR / 'installation_complete').exists():
    (ENV_DIR / 'conda-meta').mkdir(parents=True, exist_ok=True)
    menv = dict(os.environ, MAMBA_ROOT_PREFIX=str(ENV_DIR),
                CONDA_PKGS_DIRS=str(RUNTIME_DIR / 'packages'), XDG_CACHE_HOME=str(RUNTIME_DIR / 'cache'))
    command = [str(mamba), 'install', '-y', '-p', str(ENV_DIR), '-c', 'conda-forge',
               'python=3.11', 'ambertools=26.0', 'openmm=8.2.0', 'cuda-version=12.4',
               'pdbfixer=1.12', 'parmed=4.3.1', 'biopython=1.88', 'scipy=1.13.1',
               'numpy=1.26.4', 'openpyxl=3.1.5']
    subprocess.run(command, env=menv, check=True)
    (ENV_DIR / 'installation_complete').write_text('OK')
worker_env = dict(os.environ)
worker_env['PATH'] = str(ENV_DIR / 'bin') + os.pathsep + worker_env.get('PATH', '')
worker_env['AMBERHOME'] = str(ENV_DIR)
worker_env['PYTHONPATH'] = str(ASSETS / 'src') + os.pathsep + str(ASSETS / 'rna')

def worker(source):
    p = subprocess.Popen([str(ENV_DIR / 'bin/python'), '-u', '-c', source], env=worker_env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout: print(line, end='')
    code = p.wait()
    if code: raise RuntimeError('Worker failed with exit status ' + str(code))
print('Environment ready; assets are bundled. First installation can take several minutes.')


In [ ]:
# Verify CUDA with a real OpenMM context. There is no silent CPU fallback.
subprocess.run(['nvidia-smi', '-L'], check=True)
worker("""from openmm import System, Context, VerletIntegrator, Platform
s=System(); s.addParticle(12.0)
i=VerletIntegrator(0.001)
p=Platform.getPlatformByName('CUDA')
c=Context(s,i,p)
print('OpenMM platform:', c.getPlatform().getName())
""")


In [ ]:
# Build the common human template once. The uploaded PDB is never modified.
# Met10 is restored from human PABPC1 numbering, and a 5′-OH is used.
import hashlib, json
source_pdb = INPUTS / 'temp_complex.pdb'
source_hash = hashlib.sha256(source_pdb.read_bytes()).hexdigest()
metadata_path = ROOT / 'template_source.json'
template_path = ROOT / 'template.pdb'
if template_path.exists():
    if not metadata_path.exists() or json.loads(metadata_path.read_text()).get('source_sha256') != source_hash:
        raise RuntimeError('An existing template does not match this PDB. Choose a new project folder.')
    print('Using previously prepared template:', template_path)
else:
    worker(f"""from template import prepare_template
import json, pathlib, shutil
result=prepare_template({str(source_pdb)!r}, '/content/pabpc1_pose02_template_work', platform='CUDA')
shutil.copy2(result['path'], {str(template_path)!r})
pathlib.Path({str(metadata_path)!r}).write_text(json.dumps({{'source_sha256':{source_hash!r},'result':result}},indent=2))
print('Template diagnostics:', result)
""")


In [ ]:
# Prepare all modified rows. Duplicate designs are computed once and mapped back.
# This will run until complete or until Colab interrupts the runtime.
worker(f"""from workflow import make_index, write_index, run_batch
rows,cfg=make_index({str(INPUTS/'Muller_benchmark.xlsx')!r}, {str(INPUTS/'Muller_Position_Top_Two_Chemical_Modification_Library.xlsx')!r}, {str(ROOT/'template.pdb')!r})
write_index({str(ROOT)!r},rows,cfg)
run_batch({str(ROOT)!r},'/content/pabpc1_pose02_case_cache',{str(ROOT/'template.pdb')!r},
          {str(ASSETS/'libraries')!r},{str(ASSETS/'modxna')!r},rows,'variant',platform='CUDA')
""")


## Interpreting results

A more negative variant-minus-parent score indicates a more favorable **modeled score** under this one-structure protocol. Compare each candidate with its own control first. Absolute scores, small differences, and differences between unrelated parents are not calibrated binding free energies. Gaps in the CSV mean missing, failed or unconverged cases, not zero binding.

The revised 34-sequence workbook has no sequence-specific measured binding enrichments. It cannot establish a correlation with experiment until measured values are obtained. Müller’s measurements used longer 50-nt UTRs, and that study’s position −1 convention can differ from the annotated junction used here. The full calculation must be treated as exploratory until validated on an appropriate experiment.

Check `failures/` and each `minimization.json` on Drive. Rerunning skips completed cases and preserves existing unconverged results. Files are gzip-compressed; the scoring notebook decompresses them automatically. The first completed-case runtime is only a rough guide, not a completion guarantee.
